# Western-U.S. quick threshold examples

A bounded, exploratory workflow for testing NDWI/NDVI calibration on additional HydroLAKES features. Samples are transferred directly into pandas with `ee.data.computeFeatures`; this notebook creates no Earth Engine output tasks.

## 1. Initialize from ambient ADC

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'configs' / 'western_us.yaml').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'configs' / 'western_us.yaml').exists():
    raise FileNotFoundError('Run from the repository or notebooks directory.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from regional_analysis.config import load_config
from regional_analysis.discover_lakes import initialize_earth_engine
from regional_analysis.quick_examples import (
    QUICK_EXAMPLE_SETS,
    fast_mode_notice,
    run_lake_reference,
    run_quick_examples,
)
from regional_analysis.quick_sampling import (
    QuickSamplingSettings,
    build_buffered_lake_aoi,
    load_candidate_inventory,
    select_lake_specs,
    summary_dataframe,
)

config = load_config(REPO_ROOT / 'configs' / 'western_us.yaml')
inventory = load_candidate_inventory(
    REPO_ROOT / 'data' / 'regional' / 'western_hydrolakes_candidates.csv'
)
ee, project = initialize_earth_engine()
print(f'Earth Engine project: {project}')

## 2. Select a quick-example set and settings

In [ ]:
SET_NAME = 'smoke'  # change to 'saline6' only after smoke tests are stable
REFERENCES = ('JRC', 'DW')  # add 'OPERA' only after JRC + DW succeeds
settings = QuickSamplingSettings(
    buffer_km=5,
    samples_per_image=20,
    analysis_end=pd.Timestamp('2026-08-09').date(),
)
lakes = select_lake_specs(inventory, QUICK_EXAMPLE_SETS[SET_NAME])
print(fast_mode_notice(settings))
for reference in REFERENCES:
    start, end = settings.reference_window(reference)
    print(f'{reference}: {start}..{end}; ±{settings.join_window_days(reference)} day join')
display(pd.DataFrame([lake.__dict__ for lake in lakes]))

## 3. HydroLAKES polygons and buffered AOIs

In [ ]:
import geemap

qa_map = geemap.Map(center=[39.5, -118.0], zoom=5)
for lake in lakes:
    feature, aoi = build_buffered_lake_aoi(
        ee, config.earth_engine.hydrolakes_asset, lake, settings.buffer_km
    )
    polygon_layer = ee.FeatureCollection([feature]).style(
        color='2166ac', fillColor='2166ac55', width=2
    )
    buffer_layer = ee.FeatureCollection([ee.Feature(aoi)]).style(
        color='d95f02', fillColor='00000000', width=1
    )
    qa_map.addLayer(polygon_layer, {}, f'{lake.lake_name} polygon')
    qa_map.addLayer(buffer_layer, {}, f'{lake.lake_name} + buffer')
qa_map

## 4. Test 1 — Lake Abert × JRC

Run this single combination first and inspect the actual DataFrame schema and diagnostics before expanding the request.

In [ ]:
lake_abert = select_lake_specs(inventory, [776])[0]
abert_jrc = run_lake_reference(
    ee,
    config.earth_engine.hydrolakes_asset,
    lake_abert,
    'JRC',
    settings,
)
print('Raw computeFeatures columns:', abert_jrc.raw_columns)
display(pd.Series(abert_jrc.diagnostics, name='value').to_frame())
display(abert_jrc.samples.head())
display(summary_dataframe([abert_jrc.summary]))

## 5. Run the selected quick-example set

Combinations run sequentially. Intermediate samples remain in memory and are not written.

In [ ]:
quick_run = run_quick_examples(
    ee,
    config.earth_engine.hydrolakes_asset,
    lakes,
    REFERENCES,
    settings,
)
display(quick_run.summary)

## 6. Simple threshold and held-out-IoU comparisons

In [ ]:
successful = quick_run.summary.query("status == 'ok'")
display(successful.pivot(index='Lake_name', columns='reference', values='threshold_mean'))
display(successful.pivot(index='Lake_name', columns='reference', values='test_iou'))

if not successful.empty:
    axes = successful.plot.bar(
        x='Lake_name', y=['threshold_mean', 'test_iou'], subplots=True,
        figsize=(8, 6), title=['Threshold', 'Held-out IoU'], legend=False
    )

## 7. Notes

These are exploratory examples, not representative western lakes and not final regional thresholds. HydroLAKES polygon-plus-buffer AOIs differ from the historical rectangular Great Salt Lake and Lake Abert AOIs, so exact baseline reproduction is not expected. Dynamic World snow/ice is excluded pixel-by-pixel instead of rejecting whole snowy scenes. Same-time reference tiles are mosaicked for Dynamic World and for the GEE-native April-2023-present OPERA collection. No plugin defaults are changed, and entire-lake transfer validation belongs to later work.